<a href="https://colab.research.google.com/github/sruthi-kurra/llm-distillation/blob/main/notebooks/01_teacher_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Install dependencies

In [ ]:
!pip install transformers datasets evaluate rouge_score sentencepiece accelerate -q

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All dependencies installed!")

## Cell 2 — Load BART Large CNN teacher model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading BART Large CNN teacher model...")
teacher = AutoModelForSeq2SeqLM.from_pretrained(model_name)
teacher = teacher.cuda()
teacher.eval()

total_params = sum(p.numel() for p in teacher.parameters())
print(f"\nTeacher model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Model size: {total_params / 1e6:.1f}M parameters")
print(f"Device: {next(teacher.parameters()).device}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Cell 3 — Load CNN/DailyMail dataset (20k subset)

In [ ]:
from datasets import load_dataset

print("Loading CNN/DailyMail dataset...")
full_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Take subsets for faster iteration
train_dataset = full_dataset['train'].select(range(20000))
val_dataset   = full_dataset['validation'].select(range(1000))
test_dataset  = full_dataset['test'].select(range(500))

print(f"\nDataset subsets:")
print(f"Train: {len(train_dataset):,} articles")
print(f"Validation: {len(val_dataset):,} articles")
print(f"Test: {len(test_dataset):,} articles")

# Preview one example
example = test_dataset[0]
print(f"\nExample article (first 300 chars):")
print(example['article'][:300])
print(f"\nReference summary:")
print(example['highlights'])

## Cell 4 — Evaluate teacher ROUGE baseline

In [ ]:
import evaluate
import torch

rouge = evaluate.load("rouge")

def generate_summary(article, max_input_length=512, max_output_length=128):
    inputs = tokenizer(
        article,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        summary_ids = teacher.generate(
            inputs["input_ids"],
            num_beams=4,
            max_length=max_output_length,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Evaluating teacher on 100 test examples...")
predictions = []
references  = []

eval_subset = test_dataset.select(range(100))

for i, example in enumerate(eval_subset):
    pred = generate_summary(example['article'])
    predictions.append(pred)
    references.append(example['highlights'])
    if (i+1) % 20 == 0:
        print(f"Processed {i+1}/100...")

results = rouge.compute(predictions=predictions, references=references)

print(f"\n=== Teacher (BART Large CNN) ROUGE Scores ===")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")

teacher_rouge = results
print("\nTeacher baseline saved!")

import json, os

os.makedirs("results", exist_ok=True)

metrics = {
    "model": "facebook/bart-large-cnn",
    "parameters_millions": 406,
    "dataset": "CNN/DailyMail",
    "eval_samples": 100,
    "metrics": {
        "rouge1": float(results["rouge1"]),
        "rouge2": float(results["rouge2"]),
        "rougeL": float(results["rougeL"])
    }
}

with open("results/teacher_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved teacher metrics.")

In [ ]:
from google.colab import files
files.download('results/teacher_metrics.json')